In [ ]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

!git clone https://github.com/Team-Isaac-Polito/computer-vision || (cd computer-vision && git pull)

Cloning into 'computer-vision'...
remote: Enumerating objects: 447, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 447 (delta 71), reused 156 (delta 33), pack-reused 211 (from 1)
Receiving objects: 100% (447/447), 79.24 MiB | 11.67 MiB/s, done.
Resolving deltas: 100% (132/132), done.


In [ ]:

!cd computer-vision && git checkout robust_object_detection
!cd computer-vision && git pull

Branch 'robust_object_detection' set up to track remote branch 'robust_object_detection' from 'origin'.
Switched to a new branch 'robust_object_detection'
Already up to date.


In [ ]:

!mkdir -p /content/dataset
!unzip -o /content/drive/MyDrive/datasets/archive.zip -d /content/dataset

流式输出内容被截断，只能显示最后 5000 行内容。
  inflating: /content/dataset/valid/images/006803_jpg.rf.fb7851a9e5bac94bf5a0077b4368498c.jpg  
  inflating: /content/dataset/valid/images/006804_jpg.rf.97bb7a80bde3a12c7459aa914238e9ba.jpg  
  inflating: /content/dataset/valid/images/006805_jpg.rf.0bd999bb7ea36b4e41e23d2e2c5d331f.jpg  
  inflating: /content/dataset/valid/images/006806_jpg.rf.308613b85baae930099a33b6de997e41.jpg  
  inflating: /content/dataset/valid/images/006807_jpg.rf.468f4ea4a6c3061e0df94e0a6b421ccd.jpg  
  inflating: /content/dataset/valid/images/006808_jpg.rf.ff0d1667fd0188370de2bfa92d5b3512.jpg  
  inflating: /content/dataset/valid/images/006809_jpg.rf.1c018d9e80f676cd5f442cf102002b23.jpg  
  inflating: /content/dataset/valid/images/006810_jpg.rf.7826a7754c91c0df60758d17cdf44087.jpg  
  inflating: /content/dataset/valid/images/006811_jpg.rf.c97ac4643321424e544aaf94419c44c7.jpg  
  inflating: /content/dataset/valid/images/006812_jpg.rf.ff7277a7d90ec8b9f34f0f04341b0e9d.jpg  
  inflating: 

In [ ]:

!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00


In [ ]:
# ----------------------------------------------------
# 4.Train4
# ----------------------------------------------------
from ultralytics import YOLO
import torch

# A. Define configuration
# ----------------------------------------------------
# Using YOLO11s (Small) model
MODEL_PATH = "yolo11s.pt"
DATA_PATH = "/content/dataset/data.yaml"
DEVICE = "0"

# B. Model Loading and Training
# ----------------------------------------------------
model = YOLO(MODEL_PATH)

print(f"\n🚀 Starting Train4 (Ultimate Merged) with YOLO11s on {DEVICE} ...\n")

# Begin training
results = model.train(
    # ====== Basic configuration ======
    data=DATA_PATH,
    imgsz=640,
    epochs=80,          # Increased to 80 rounds, with a low learning rate
    batch=64,           
    device=DEVICE,
    pretrained=True,    

    # ====== Stability (Solving Loss Fluctuations) ======
    optimizer='auto',
    lr0=0.005,          # Lower the initial learning rate to resolve start-up oscillations
    lrf=0.0005,         # Reduce final learning rate and improve accuracy
    cos_lr=True,        # Start cosine annealing
    warmup_epochs=5,    # Extended warm-up time
    label_smoothing=0.1,# Strongly suppresses overfitting and loss spikes
    weight_decay=0.0005,

    # ====== Hardware acceleration (for A100) ======
    cache=True,         # Enable memory caching to speed up training
    compile=True,       # PyTorch 2.0 compilation speedup
    workers=8,          # Ensure smooth data loading

    # ====== Data augmentation and fine-tuning ======
    augment=True,
    mixup=0.1,
    cutmix=0.3,         # 0.5 -> 0.3, more favorable for S-model.
    mosaic=1.0,
    close_mosaic=10,    # The mosaic was turned off in the final 10 rounds to improve accuracy.

    # ====== Geometric transformation ======
    translate=0.2,      # Improve position adaptability
    scale=0.7,          # Improve size adaptability
    perspective=0.001,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,

    # ====== Verification and save ======
    val=True,
    save=True,
    # Save to Google Drive
    project='/content/drive/MyDrive/runs/detect',
    name='train4_yolo11s'
)

print("\n--- Train finish.  ---")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

🚀 Starting Train4 (Ultimate Merged) with YOLO11s on 0 ...

WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (NVIDIA A100-SXM4-80GB, 81222MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=True, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.3, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, fli

In [ ]:
print("\n--- 训练完成结果摘要 ---")
   
for metric_name, value in results.results_dict.items():
        print(f'{metric_name}: {value:.4f}')


--- 训练完成结果摘要 ---
metrics/precision(B): 0.9656
metrics/recall(B): 0.9393
metrics/mAP50(B): 0.9763
metrics/mAP50-95(B): 0.7753
fitness: 0.7753
